In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.__version__

'3.0.6'

In [3]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-09-29 00:24:19--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.1’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.03s   

2026-09-29 00:24:19 (21.6 MB/s) - ‘car_fuel_efficiency_2026.csv.1’ saved [635180/635180]



In [5]:
data=pd.read_csv("car_fuel_efficiency_2026.csv")

In [6]:
data.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Q2. Records count

How many records are in the dataset?

- 5000
- 9000
- 10000
- 15000


In [7]:
len(data)

10000


## Q3. Fuel types

How many fuel types are presented in the dataset?

- 1
- 2
- 3
- 4

In [8]:
data.fuel_type.nunique()

3

## Q4. Missing values

How many columns in the dataset have missing values?

- 0
- 1
- 2
- 3
- 4


In [9]:
sum(data.isna().sum()>0)

2

## Q5. Max fuel efficiency

What's the maximum fuel efficiency of cars from Asia?

- 21.2
- 31.2
- 41.2
- 51.2


In [10]:
data[data.origin=="Asia"].fuel_efficiency_mpg.max()

np.float64(41.2)

In [11]:
data2 = data[~(data.isna())]

In [12]:
data2[data2.origin=="Asia"].fuel_efficiency_mpg.max()

np.float64(41.2)

# Q6. Median value of horsepower

1. Find the median value of the `horsepower` column in the dataset.
2. Next, calculate the most frequent value of the same `horsepower` column.
3. Use the `fillna` method to fill the missing values in the `horsepower` column with the most frequent value from the previous step.
4. Now, calculate the median value of `horsepower` once again.

In [13]:
data.head(2)

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3


In [14]:
# Find the median value
data.horsepower.median()

np.float64(254.0)

In [15]:
# Fillna with most frequent value
data.horsepower.value_counts()#.index[0]

horsepower
252.0    174
246.0    168
256.0    168
259.0    165
253.0    165
        ... 
322.0      1
330.0      1
175.0      1
178.0      1
320.0      1
Name: count, Length: 153, dtype: int64

In [16]:
data["horsepower_2"] = data["horsepower"].fillna(data.horsepower.value_counts().index[0])

In [17]:
# Find the median value of imputed columns
data.horsepower_2.median()

np.float64(252.0)

## Q7. Sum of weights

1. Select all the cars from Asia
2. Select only columns `vehicle_weight` and `model_year`
3. Select the first 7 values
4. Get the underlying NumPy array. Let's call it `X`.
5. Compute matrix-matrix multiplication between the transpose of `X` and `X`. To get the transpose, use `X.T`. Let's call the result `XTX`.
6. Invert `XTX`.
7. Create an array `y` with values `[1100, 1300, 800, 900, 1000, 1100, 1200]`.
8. Multiply the inverse of `XTX` with the transpose of `X`, and then multiply the result by `y`. Call the result `w`.
9. What's the sum of all the elements of the result?


In [18]:
data_asia = data[data.origin=='Asia'][["vehicle_weight","model_year"]].iloc[:7]

In [19]:
X = np.array(data_asia)

In [20]:
X

array([[4240, 1996],
       [4310, 1992],
       [4230, 1997],
       [4420, 1978],
       [3950, 2007],
       [4750, 2021],
       [4450, 2019]])

In [21]:
XTX =  np.dot(X.T,X)
XTX


array([[131950500,  60750580],
       [ 60750580,  28041424]])

In [22]:
XTX_inv = np.linalg.inv(XTX)
XTX_inv

array([[ 2.96830537e-06, -6.43071025e-06],
       [-6.43071025e-06,  1.39675281e-05]])

In [23]:
y = [1100, 1300, 800, 900, 1000, 1100, 1200]

In [24]:
w = np.dot(np.dot(XTX_inv,X.T),y)
w

array([0.13644777, 0.2327492 ])

In [25]:
w.sum()

np.float64(0.3691969690492552)

In [26]:
def augment_tabular_data(df, target_multiplier=10, noise_level=0.05, seed=42):
  np.random.seed(seed)
  n_original = len(df)
  n_target = n_original * target_multiplier

  # 1. Amostragem com reposição para atingir o tamanho desejado
  augmented_df = df.sample(n=n_target, replace=True).reset_index(drop=True)

  # 2. Separar colunas numéricas e categóricas
  num_cols = augmented_df.select_dtypes(include=[np.number]).columns
  cat_cols = augmented_df.select_dtypes(include=["object", "category"]).columns

  # 3. Adicionar ruído controlado às colunas numéricas para criar variação realista
  for col in num_cols:
    std_dev = augmented_df[col].std()
    # Adiciona um pequeno ruído gaussiano proporcional ao desvio padrão da coluna
    noise = np.random.normal(0, noise_level * std_dev, size=n_target)
    augmented_df[col] = augmented_df[col] + noise

  # Opcional: Se houver colunas categóricas, podemos introduzir pequenas variações ou mantê-las como amostras fiéis.
  # Aqui mantemos as amostras categóricas fiéis à distribuição original via amostragem.

  print(
      f"Dataset original: {n_original} linhas | Dataset aumentado: {len(augmented_df)} linhas"
  )
  return augmented_df

In [30]:
# Gerando o dataset 10X maior
df_10x = augment_tabular_data(data, target_multiplier=100, noise_level=0.03)

print(df_10x.head())

/tmp/ipykernel_1869/2534222115.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = augmented_df.select_dtypes(include=["object", "category"]).columns


Dataset original: 10000 linhas | Dataset aumentado: 1000000 linhas
    model_year  origin fuel_type        drivetrain  num_doors  \
0  1996.972561  Europe    Diesel  Rear-wheel drive   1.989955   
1  2013.469569  Europe    Diesel  Rear-wheel drive   4.034822   
2  2012.927539  Europe  Gasoline   All-wheel drive   2.985192   
3  1999.438935    Asia  Gasoline   All-wheel drive   3.995985   
4  1988.551573  Europe    Diesel  Rear-wheel drive   3.041149   

   engine_displacement  num_cylinders  horsepower  vehicle_weight  \
0          2065.418668       6.005089  273.147413     3967.690891   
1          2445.522947       7.007207  306.803749     4516.498569   
2          2425.365186       6.005212  245.627776     4654.806419   
3          2157.179347       6.005159  231.616810     4175.084952   
4          2108.665191       4.966979  256.317629     4125.610533   

   acceleration  fuel_efficiency_mpg  horsepower_2  
0     19.481978            34.858097    274.458104  
1     19.080548      

In [32]:
df_10x.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,horsepower_2
0,1996.972561,Europe,Diesel,Rear-wheel drive,1.989955,2065.418668,6.005089,273.147413,3967.690891,19.481978,34.858097,274.458104
1,2013.469569,Europe,Diesel,Rear-wheel drive,4.034822,2445.522947,7.007207,306.803749,4516.498569,19.080548,30.436521,307.540568
2,2012.927539,Europe,Gasoline,All-wheel drive,2.985192,2425.365186,6.005212,245.627776,4654.806419,19.356432,30.096602,246.582158
3,1999.438935,Asia,Gasoline,All-wheel drive,3.995985,2157.179347,6.005159,231.616810,4175.084952,18.346442,25.545062,230.956763
4,1988.551573,Europe,Diesel,Rear-wheel drive,3.041149,2108.665191,4.966979,256.317629,4125.610533,17.303602,30.753704,257.430207


In [36]:
df_10x.isna().sum()

model_year                 0
origin                     0
fuel_type                  0
drivetrain                 0
num_doors                  0
engine_displacement        0
num_cylinders              0
horsepower             87876
vehicle_weight             0
acceleration           26559
fuel_efficiency_mpg        0
horsepower_2               0
dtype: int64

In [ ]:
X = df_10x[["vehicle_weight", "model_year", "fuel_type"]]
X["fuel_type"] = X["fuel_type"].astype("category")

y = df_10x["fuel_efficiency_mpg"]

In [39]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [42]:
model = XGBRegressor(objective='reg:squarederror', random_state=123,enable_categorical=True)
model.fit(X_train, y_train)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:fuel_type: str